In [34]:
import pandas as pd
import numpy as np
import requests
import boto3
from datetime import datetime

In [2]:
parks_url = 'https://queue-times.com/parks.json'

In [3]:
response = requests.get(parks_url)

In [4]:
print(response.status_code)

200


In [5]:
data = response.json()


In [6]:
print(data)

[{'id': 11, 'name': 'Cedar Fair Entertainment Company', 'parks': [{'id': 57, 'name': "California's Great America", 'country': 'United States', 'continent': 'North America', 'latitude': '37.397799', 'longitude': '-121.974717', 'timezone': 'America/Los_Angeles'}, {'id': 58, 'name': "Canada's Wonderland", 'country': 'Canada', 'continent': 'North America', 'latitude': '43.843', 'longitude': '-79.539', 'timezone': 'America/Toronto'}, {'id': 59, 'name': 'Carowinds', 'country': 'United States', 'continent': 'North America', 'latitude': '35.1045', 'longitude': '-80.9394', 'timezone': 'America/New_York'}, {'id': 50, 'name': 'Cedar Point', 'country': 'United States', 'continent': 'North America', 'latitude': '41.4822', 'longitude': '-82.6835', 'timezone': 'America/New_York'}, {'id': 69, 'name': 'Dorney Park', 'country': 'United States', 'continent': 'North America', 'latitude': '40.577942', 'longitude': '-75.531528', 'timezone': 'America/New_York'}, {'id': 62, 'name': 'Kings Dominion', 'country'

In [7]:
filtered_data = [x for x in data if 'Disney' in x['name']]
#print(filtered_data)
# results:
# 16 Disneyland
# 17 Disney Calfiornia Adventure

In [8]:
# timestamps are in UTC
disneyland_wait_times_url = 'https://queue-times.com/parks/16/queue_times.json'
dca_wait_times_url = 'https://queue-times.com/parks/17/queue_times.json'
disneyland_response = requests.get(disneyland_wait_times_url)
dca_response = requests.get(dca_wait_times_url)

In [9]:
print(disneyland_response.status_code)

200


In [10]:
disneyland_data = disneyland_response.json()
dca_data = dca_response.json()

In [11]:
print(disneyland_data)

{'lands': [{'id': 109, 'name': 'Adventureland', 'rides': [{'id': 12428, 'name': 'Adventureland Treehouse inspired by Walt Disney’s Swiss Family Robinson', 'is_open': True, 'wait_time': 0, 'last_updated': '2025-04-22T01:31:19.000Z'}, {'id': 326, 'name': 'Indiana Jones™ Adventure', 'is_open': False, 'wait_time': 0, 'last_updated': '2025-04-22T01:31:19.000Z'}, {'id': 296, 'name': 'Jungle Cruise', 'is_open': True, 'wait_time': 15, 'last_updated': '2025-04-22T01:31:19.000Z'}, {'id': 288, 'name': "Walt Disney's Enchanted Tiki Room", 'is_open': True, 'wait_time': 0, 'last_updated': '2025-04-22T01:31:19.000Z'}]}, {'id': 110, 'name': 'Bayou Country', 'rides': [{'id': 304, 'name': "Davy Crockett's Explorer Canoes", 'is_open': False, 'wait_time': 0, 'last_updated': '2025-04-22T01:31:19.000Z'}, {'id': 306, 'name': 'The Many Adventures of Winnie the Pooh', 'is_open': True, 'wait_time': 15, 'last_updated': '2025-04-22T01:31:19.000Z'}, {'id': 14168, 'name': "Tiana's Bayou Adventure", 'is_open': True,

In [41]:
# convert to dataframe and combine datasets
dl_rides_df = pd.json_normalize(disneyland_data['lands'], 'rides')
dca_rides_df = pd.json_normalize(dca_data['lands'], 'rides')

combined_df = pd.concat([dl_rides_df, dca_rides_df], ignore_index = True)

In [42]:
print(combined_df)

       id                                               name  is_open  \
0   12428  Adventureland Treehouse inspired by Walt Disne...     True   
1     326                           Indiana Jones™ Adventure    False   
2     296                                      Jungle Cruise     True   
3     288                  Walt Disney's Enchanted Tiki Room     True   
4     304                    Davy Crockett's Explorer Canoes    False   
..    ...                                                ...      ...   
78    300                                  Jumpin' Jellyfish     True   
79    311                       Pixar Pal-A-Round - Swinging     True   
80   5557                   Pixar Pal-A-Round – Non-Swinging     True   
81    313                            Toy Story Midway Mania!     True   
82  13959                                    The Bakery Tour     True   

    wait_time              last_updated  
0           0  2025-04-22T01:31:19.000Z  
1           0  2025-04-22T01:31:19.000Z

In [43]:
# convert UTC to PST
combined_df['last_updated'] = pd.to_datetime(combined_df['last_updated'])
combined_df['last_updated_pst'] = combined_df['last_updated'].dt.tz_convert('US/Pacific')
combined_df.drop('last_updated', axis=1, inplace=True)
combined_df.rename(columns={'last_updated_pst':'last_updated'}, inplace=True)
print(combined_df.head())

      id                                               name  is_open  \
0  12428  Adventureland Treehouse inspired by Walt Disne...     True   
1    326                           Indiana Jones™ Adventure    False   
2    296                                      Jungle Cruise     True   
3    288                  Walt Disney's Enchanted Tiki Room     True   
4    304                    Davy Crockett's Explorer Canoes    False   

   wait_time              last_updated  
0          0 2025-04-21 18:31:19-07:00  
1          0 2025-04-21 18:31:19-07:00  
2         15 2025-04-21 18:31:19-07:00  
3          0 2025-04-21 18:31:19-07:00  
4          0 2025-04-21 18:31:19-07:00  


In [44]:
# Initialize the DynamoDB resource
dynamodb = boto3.resource('dynamodb', region_name='us-east-1')

# Specify the table name
table = dynamodb.Table('disney_wait_times')

In [58]:
from botocore.exceptions import NoCredentialsError

try:
    session = boto3.Session()
    credentials = session.get_credentials()
    print("Credentials loaded:", credentials)
except NoCredentialsError:
    print("No credentials found!")


Credentials loaded: <botocore.credentials.Credentials object at 0x00000299E0ED95B0>


In [46]:
table = dynamodb.Table('disney_wait_times')

In [53]:
with table.batch_writer() as batch:
    for _, row in combined_df.iterrows():
        item = {}

        # Ensure partition key and sort key are included
        item['id'] = int(row['id'])  # Convert to string if needed
        item['last_updated'] = row['last_updated'].isoformat()  # Convert datetime to ISO format

        # Add other attributes
        for col, val in row.items():
            if col not in ['id', 'last_updated']:  # Avoid duplicate key assignments
                if isinstance(val, datetime):
                    item[col] = val.isoformat()
                elif isinstance(val, float):
                    item[col] = Decimal(str(val))
                else:
                    item[col] = val

        batch.put_item(Item=item)

print('Data added successfully')


Data added successfully


In [65]:
response = table.scan()
items_df = pd.DataFrame(response['Items'])
print(items_df)

                 last_updated wait_time     id  \
0   2025-04-21T18:31:19-07:00        15    286   
1   2025-04-21T18:31:19-07:00         0    304   
2   2025-04-21T18:31:19-07:00        15    273   
3   2025-04-21T18:31:19-07:00         0  11980   
4   2025-04-21T18:31:39-07:00        25  13961   
..                        ...       ...    ...   
79  2025-04-21T18:31:19-07:00         0  14326   
80  2025-04-21T18:31:19-07:00        15    306   
81  2025-04-21T18:31:39-07:00        30    311   
82  2025-04-21T18:31:19-07:00        55  11526   
83  2025-04-21T18:31:39-07:00        40    295   

                                      name is_open  
0     Star Tours - The Adventures Continue    True  
1          Davy Crockett's Explorer Canoes   False  
2            Buzz Lightyear Astro Blasters    True  
3                 Goofy's How-to-Play Yard    True  
4             Luigi's Rollickin' Roadsters    True  
..                                     ...     ...  
79    Tiana's Bayou Adventur

In [67]:
response2 = table.query(
    KeyConditionExpression=boto3.dynamodb.conditions.Key('id').eq(295)  # Replace with actual ID
)
items_df2 = pd.DataFrame(response2['Items'])
print(items_df2)

                last_updated wait_time   id                     name  is_open
0  2025-04-21T18:31:39-07:00        40  295  Radiator Springs Racers     True
